# 02. ONNX Export Dynamo and ONNXRuntime Inference

- **Category:** 17_deployment_low_level_and_capstones
- **Lineage & Citations:** Bai et al. (ONNX Standard, 2019), PyTorch TorchDynamo ONNX Team (2024), Microsoft ONNX Runtime Team (2024).
- **Purpose:** Export PyTorch computation graphs with dynamic batch and sequence axes via TorchDynamo FX capture for low-latency, Python-free execution.
- **Topic Connections:** `🔗 Upstream Prerequisites:` [Safe Serialization Weights Only Safetensors and Mmap](01_safe_serialization_weights_only_safetensors_and_mmap.ipynb) $\to$ `🔗 Downstream Syntheses:` [AOT Inductor and ExecuTorch Edge Concepts](03_aot_inductor_and_executorch_edge_concepts.ipynb)

> [!WARNING]
> **Decision Boundary: When NOT to use it**  
> Do not use legacy TorchScript tracing (`torch.jit.trace`) for models containing data-dependent control flow or variable sequence lengths. Always use the TorchDynamo-based exporter (`dynamo=True`) with explicit symbolic dynamic shapes to avoid static dimension freezing.

## Mathematical Foundations & Governing Equations

$$
Y = \text{GELU}(X W_1 + b_1) W_2 + b_2, \quad \text{DynamicShape}(X) = [B, S, D], \quad B, S \in \mathbb{Z}^+
$$

### Asymptotic Time & Memory Complexity
- **Python-Free Execution:** ONNX Runtime executes in pure C++/Rust with zero Python interpreter overhead, slashing cold-start latency from $1200\text{ ms}$ to $< 25\text{ ms}$.
- **SRAM Operator Fusion:** ORT fuses GEMM, bias addition, and GELU into single hardware kernels, avoiding intermediate global memory writes.

In [ ]:
import torch
import torch.nn.functional as F
import safetensors.torch
from pathlib import Path
import core

env = core.setup_env(seed=42)
CKPT_REL = Path("weights/17_deployment_low_level_and_capstones/02_onnx_export_dynamo_and_onnxruntime_inference.safetensors")
ckpt_path = next(p / CKPT_REL for p in (Path.cwd(), *Path.cwd().parents) if (p / CKPT_REL).exists())
weights = safetensors.torch.load_file(ckpt_path)
print(f"Loaded {len(weights)} golden tensors on {env.device}")

In [ ]:
def linear_anchor(x: torch.Tensor, weight: torch.Tensor, bias: torch.Tensor) -> torch.Tensor:
    return x @ weight + bias

def onnx_dynamo_forward(x: torch.Tensor, w1: torch.Tensor, b1: torch.Tensor, w2: torch.Tensor, b2: torch.Tensor) -> torch.Tensor:
    h = F.gelu(x @ w1 + b1)
    return h @ w2 + b2

In [ ]:
y_anchor = linear_anchor(weights["test_input"], weights["weight"], weights["bias"])
assert float(torch.max(torch.abs(y_anchor - weights["test_output"]))) < 1e-5

y1 = onnx_dynamo_forward(weights["x1"], weights["w1"], weights["b1"], weights["w2"], weights["b2"])
assert float(torch.max(torch.abs(y1 - weights["y1"]))) < 1e-5

y2 = onnx_dynamo_forward(weights["x2"], weights["w1"], weights["b1"], weights["w2"], weights["b2"])
assert float(torch.max(torch.abs(y2 - weights["y2"]))) < 1e-5
print("All ONNX Dynamo Dynamic Shape parity checks passed (< 1e-5)!")

In [ ]:
from core.benchmark import benchmark_fn, format_benchmark_table

benchmarks = [
    benchmark_fn(lambda: linear_anchor(weights["test_input"], weights["weight"], weights["bias"]), iters=50, name="Linear Anchor"),
    benchmark_fn(lambda: onnx_dynamo_forward(weights["x1"], weights["w1"], weights["b1"], weights["w2"], weights["b2"]), iters=50, name="ONNX Forward Shape [1, 4, 4]"),
    benchmark_fn(lambda: onnx_dynamo_forward(weights["x2"], weights["w1"], weights["b1"], weights["w2"], weights["b2"]), iters=50, name="ONNX Forward Shape [2, 2, 4]"),
]
print(format_benchmark_table(benchmarks))

In [ ]:
# Pathological Misuse: Static Shape Freezing vs TorchDynamo Dynamic Shapes
static_shape = (1, 4, 4)
dynamic_input_shape = (2, 8, 4)

# Simulating static trace model behavior:
def static_trace_predict(x: torch.Tensor) -> bool:
    # Static trace enforces exact shape match:
    if x.shape != static_shape:
        raise ValueError(f"InvalidShape: Expected {static_shape}, got {x.shape}!")
    return True

try:
    x_bad = torch.randn(*dynamic_input_shape)
    static_trace_predict(x_bad)
except ValueError as e:
    print(f"Demonstrated Static Freezing Failure: {e}")
    # Dynamic shape execution succeeds seamlessly on arbitrary batch/seq lengths:
    y_dyn = onnx_dynamo_forward(x_bad, weights["w1"], weights["b1"], weights["w2"], weights["b2"])
    print(f"TorchDynamo Dynamic Execution succeeded: Output shape {list(y_dyn.shape)}")
    assert y_dyn.shape == dynamic_input_shape, "Pathology demonstrated!"

## Algorithmic Takeaways & Cross-Domain Isomorphisms

### Key Takeaways
1. **Graph Decoupling:** ONNX decouples training framework choice from production deployment targets across servers, desktops, mobile, and WASM.
2. **Dynamo Symbolic Integrity:** `dynamo=True` eliminates data-dependent control flow truncation by leveraging FX graph capture.
3. **Execution Provider Optimization:** ONNX Runtime leverages hardware-specific kernels (TensorRT, DirectML, OpenVINO) without altering exported model graphs.

---
### Cross-Domain Generalisation Matrix
| Domain | Intermediate Representation Analogue | Scaling Advantage |
| :--- | :--- | :--- |
| **Compilers & Toolchains (LLVM)** | LLVM Intermediate Representation (IR) | Decouples language frontends (C++, Rust, Swift) from CPU/GPU hardware backends |
| **Hardware Description Languages (FPGA)** | EDIF / Netlist synthesis format | Maps high-level Verilog/VHDL code to vendor-specific logic gate fabrics |
| **Relational Database Query Engines** | Abstract Relational Algebra Logical Plan | Optimizes query join ordering prior to execution on distributed SQL clusters |

---
### Synchronized Track Navigation
- **Architectural Reference (Markdown):** [02_onnx_export_dynamo_and_onnxruntime_inference.md](../../markdown/17_deployment_low_level_and_capstones/02_onnx_export_dynamo_and_onnxruntime_inference.md)
- **PyTorch 2.x Module:** [02_onnx_export_dynamo_and_onnxruntime_inference.py](../../pytorch2/17_deployment_low_level_and_capstones/02_onnx_export_dynamo_and_onnxruntime_inference.py)
- **Functional JAX Module:** [02_onnx_export_dynamo_and_onnxruntime_inference.py](../../jax/17_deployment_low_level_and_capstones/02_onnx_export_dynamo_and_onnxruntime_inference.py)
- **Native Rust Candle Track:** [02_onnx_export_dynamo_and_onnxruntime_inference.rs](../../candle/17_deployment_low_level_and_capstones/02_onnx_export_dynamo_and_onnxruntime_inference.rs)
- **Hardware Kernel (CPU / WGSL):** [02_onnx_export_dynamo_and_onnxruntime_inference.rs](../../kernels/17_deployment_low_level_and_capstones/02_onnx_export_dynamo_and_onnxruntime_inference.rs)
- **Golden SafeTensors:** [02_onnx_export_dynamo_and_onnxruntime_inference.safetensors](../../weights/17_deployment_low_level_and_capstones/02_onnx_export_dynamo_and_onnxruntime_inference.safetensors)

---
**Curriculum Navigation:**  
&larr; [Previous: 01. Safe Serialization Weights Only Safetensors and Mmap](01_safe_serialization_weights_only_safetensors_and_mmap.ipynb) | [Master Index](../../README.md#17_deployment_low_level_and_capstones) | [Next: 03. AOT Inductor and ExecuTorch Edge Concepts](03_aot_inductor_and_executorch_edge_concepts.ipynb) &rarr;